# 01 — Query the Shape Catalog from DP0.2

This notebook connects to the DP0.2 Butler, retrieves galaxy shape
measurements from the `objectTable`, and explores what the pipeline
actually produces.

**Run this on the Rubin Science Platform (RSP).**

DP0.2 uses simulated LSST images from the DC2 simulation, processed
through the full LSST Science Pipeline. It includes truth tables
so we can validate our shape measurements.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from lsst.daf.butler import Butler

## 1. Connect to the DP0.2 Data Repository

In [ ]:
# DP0.2 Butler configuration on RSP
butler = Butler('dp02', collections='2.2i/runs/DP0.2')

# Check what dataset types are available
dataset_types = sorted(butler.registry.queryDatasetTypes())
shape_related = [dt for dt in dataset_types
                 if any(kw in dt.name.lower() for kw in ['object', 'coadd_meas', 'src', 'forced'])]
print("Shape-related dataset types:")
for dt in shape_related[:15]:
    print(f"  {dt.name}")

## 2. Retrieve the Object Table

The `objectTable` is the merged, calibrated catalog of all sources
across all bands. It contains shape measurements, fluxes, and flags.

We'll pick a specific tract/patch in the DC2 footprint.

In [ ]:
# Pick a tract/patch with good coverage
# DC2 tract 4226 is well-populated
tract = 4226
patch = 17

# Retrieve the objectTable for this patch
obj_table = butler.get('objectTable', tract=tract, patch=patch)

print(f"Number of objects: {len(obj_table)}")
print(f"\nColumns ({len(obj_table.columns)} total):")

# Show shape-related columns
shape_cols = [c for c in obj_table.columns if any(kw in c.lower()
              for kw in ['shape', 'hsm', 'ixx', 'iyy', 'ixy', 'e1', 'e2',
                         'extendedness', 'blendedness'])]
print("\nShape-related columns:")
for c in sorted(shape_cols):
    print(f"  {c}")

In [ ]:
# Also show flux/magnitude columns
flux_cols = [c for c in obj_table.columns if 'cmodel' in c.lower() and 'flux' in c.lower()]
print("CModel flux columns:")
for c in sorted(flux_cols):
    print(f"  {c}")

## 3. Basic Quality Selection

Not every row in the catalog is a usable galaxy. We need to apply
quality cuts to get a clean sample for weak lensing.

In [ ]:
# Start with the full table
df = obj_table
print(f"Total objects: {len(df)}")

# Cut 1: Primary detections only (no duplicates from overlapping patches)
mask = df['detect_isPrimary']
print(f"After isPrimary: {mask.sum()}")

# Cut 2: Extended sources (galaxies, not stars)
# extendedness = 1 for galaxies, 0 for stars
mask &= df['refExtendedness'] == 1
print(f"After galaxy cut: {mask.sum()}")

# Cut 3: i-band SNR > 10 (need decent photometry)
snr_col = 'i_cModelFlux' if 'i_cModelFlux' in df.columns else 'i_calibFlux'
snr_err_col = snr_col.replace('Flux', 'FluxErr')
if snr_err_col in df.columns:
    snr = df[snr_col] / df[snr_err_col]
    mask &= snr > 10
    print(f"After SNR > 10: {mask.sum()}")

# Cut 4: No shape measurement failures
# Find the HSM flag column
hsm_flag_cols = [c for c in df.columns if 'hsm' in c.lower() and 'flag' in c.lower()]
print(f"\nHSM flag columns: {hsm_flag_cols}")

for fc in hsm_flag_cols:
    if 'regauss' in fc.lower() or 'Regauss' in fc:
        mask &= ~df[fc]
        print(f"After {fc} == False: {mask.sum()}")

# Cut 5: blendedness < 10^-0.375 (Mandelbaum et al. 2018 HSC selection)
blend_col = [c for c in df.columns if 'blendedness' in c.lower()]
if blend_col:
    mask &= df[blend_col[0]] < 10**(-0.375)
    print(f"After blendedness cut: {mask.sum()}")

galaxies = df[mask].copy()
print(f"\nFinal galaxy sample: {len(galaxies)}")

## 4. Explore the Shape Measurements

In [ ]:
# Find the e1, e2 columns — naming may vary across pipeline versions
e1_candidates = [c for c in galaxies.columns if 'e1' in c.lower() and 'hsm' in c.lower()]
e2_candidates = [c for c in galaxies.columns if 'e2' in c.lower() and 'hsm' in c.lower()]
print("e1 candidates:", e1_candidates)
print("e2 candidates:", e2_candidates)

# Use the first matching pair (prefer Regauss)
e1_col = [c for c in e1_candidates if 'regauss' in c.lower() or 'Regauss' in c]
e2_col = [c for c in e2_candidates if 'regauss' in c.lower() or 'Regauss' in c]

if e1_col and e2_col:
    e1_col, e2_col = e1_col[0], e2_col[0]
else:
    # Fallback: use whatever HSM columns exist
    e1_col, e2_col = e1_candidates[0], e2_candidates[0]

print(f"\nUsing: {e1_col}, {e2_col}")

e1 = galaxies[e1_col].values
e2 = galaxies[e2_col].values

# Remove any NaN/inf
good = np.isfinite(e1) & np.isfinite(e2)
e1, e2 = e1[good], e2[good]
print(f"Galaxies with valid shapes: {len(e1)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# e1 distribution
axes[0].hist(e1, bins=60, range=(-1, 1), color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', ls='--', lw=1.5)
axes[0].axvline(np.mean(e1), color='orange', ls='-', lw=2,
                label=f'mean={np.mean(e1):.4f}')
axes[0].set_xlabel('$e_1$', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('$e_1$ distribution', fontsize=13)
axes[0].legend()

# e2 distribution
axes[1].hist(e2, bins=60, range=(-1, 1), color='forestgreen', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', ls='--', lw=1.5)
axes[1].axvline(np.mean(e2), color='orange', ls='-', lw=2,
                label=f'mean={np.mean(e2):.4f}')
axes[1].set_xlabel('$e_2$', fontsize=12)
axes[1].set_title('$e_2$ distribution', fontsize=13)
axes[1].legend()

# e1 vs e2 scatter
axes[2].hexbin(e1, e2, gridsize=40, cmap='Blues', mincnt=1)
axes[2].plot(0, 0, 'r+', ms=15, mew=2)
axes[2].set_xlabel('$e_1$', fontsize=12)
axes[2].set_ylabel('$e_2$', fontsize=12)
axes[2].set_title('$e_1$ vs $e_2$', fontsize=13)
axes[2].set_aspect('equal')
axes[2].set_xlim(-1, 1); axes[2].set_ylim(-1, 1)

plt.suptitle(f'Galaxy Shape Measurements (N={len(e1)}, tract={tract}, patch={patch})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../figures/shape_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean e1 = {np.mean(e1):.5f} ± {np.std(e1)/np.sqrt(len(e1)):.5f}")
print(f"Mean e2 = {np.mean(e2):.5f} ± {np.std(e2)/np.sqrt(len(e2)):.5f}")
print(f"RMS ellipticity = {np.sqrt(np.mean(e1**2 + e2**2)):.4f}")
print(f"\nFor a random patch, <e1> and <e2> should be close to zero.")
print(f"The RMS ~0.25-0.3 is the intrinsic shape noise.")

## 5. Size-Magnitude Diagram: Stars vs. Galaxies

In [ ]:
# Use the full (pre-cut) table to show star-galaxy separation
full = df[df['detect_isPrimary']].copy()

# Find size columns (trace of moments)
size_cols = [c for c in full.columns if 'ixx' in c.lower() or 'iyy' in c.lower()]
print("Size-related columns:", size_cols[:10])

# Try to compute trace radius from i-band moments
ixx_col = [c for c in full.columns if 'i_ixx' in c.lower() or ('ixx' in c.lower() and c.startswith('i_'))]
iyy_col = [c for c in full.columns if 'i_iyy' in c.lower() or ('iyy' in c.lower() and c.startswith('i_'))]

# Also get PSF size
psf_ixx_col = [c for c in full.columns if 'ixxpsf' in c.lower().replace('_', '')
               and c.startswith('i_')]
psf_iyy_col = [c for c in full.columns if 'iyypsf' in c.lower().replace('_', '')
               and c.startswith('i_')]

print(f"Source Ixx: {ixx_col}")
print(f"PSF Ixx: {psf_ixx_col}")

In [ ]:
# Compute trace radius and magnitude
# Adapt column names based on what's available
try:
    T_source = full[ixx_col[0]] + full[iyy_col[0]]
    T_psf = full[psf_ixx_col[0]] + full[psf_iyy_col[0]]
    trace_radius = np.sqrt(T_source / 2)  # pixels
    psf_radius = np.sqrt(T_psf / 2)
except (IndexError, KeyError):
    # Fallback: use whatever size column is available
    print("Could not find moment columns, trying alternative size columns...")
    size_cols_alt = [c for c in full.columns if 'size' in c.lower() or 'sigma' in c.lower()]
    print(f"Available: {size_cols_alt[:10]}")
    trace_radius = None

# i-band magnitude
if 'i_cModelFlux' in full.columns:
    with np.errstate(divide='ignore', invalid='ignore'):
        i_mag = -2.5 * np.log10(full['i_cModelFlux']) + 31.4  # approximate AB mag
else:
    i_mag_col = [c for c in full.columns if c.startswith('i_') and 'mag' in c.lower()]
    if i_mag_col:
        i_mag = full[i_mag_col[0]]

is_star = full['refExtendedness'] == 0
is_galaxy = full['refExtendedness'] == 1

In [ ]:
if trace_radius is not None:
    fig, ax = plt.subplots(figsize=(9, 7))

    good_star = is_star & np.isfinite(trace_radius) & np.isfinite(i_mag)
    good_gal = is_galaxy & np.isfinite(trace_radius) & np.isfinite(i_mag)

    ax.scatter(i_mag[good_gal], trace_radius[good_gal],
               s=2, alpha=0.2, c='steelblue', label='Galaxies')
    ax.scatter(i_mag[good_star], trace_radius[good_star],
               s=2, alpha=0.4, c='gold', label='Stars')
    ax.axhline(np.nanmedian(psf_radius), color='red', ls='--', lw=1.5,
               label=f'PSF radius ({np.nanmedian(psf_radius):.2f} pix)')

    ax.set_xlabel('i-band magnitude', fontsize=12)
    ax.set_ylabel('Trace radius (pixels)', fontsize=12)
    ax.set_title('Size–Magnitude Diagram (DP0.2)', fontsize=14)
    ax.set_xlim(17, 27)
    ax.set_ylim(0, 15)
    ax.invert_xaxis()
    ax.legend(fontsize=11, markerscale=5)
    plt.tight_layout()
    plt.savefig('../../figures/size_mag_dp02.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Trace radius not available — adapt column names for your pipeline version.")

## 6. Ellipticity vs. Galaxy Properties

In [ ]:
# Ellipticity magnitude
e_mag = np.sqrt(e1**2 + e2**2)

# Get matching magnitudes for the galaxy subsample
gal_subset = galaxies[good[galaxies.index.isin(galaxies.index)]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# |e| distribution
axes[0].hist(e_mag, bins=50, range=(0, 1), color='steelblue', edgecolor='white')
axes[0].axvline(np.median(e_mag), color='red', ls='--',
                label=f'median |e| = {np.median(e_mag):.3f}')
axes[0].set_xlabel('$|e|$', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Ellipticity magnitude distribution', fontsize=13)
axes[0].legend(fontsize=11)

# Position angle distribution (should be uniform for no preferred direction)
phi = 0.5 * np.arctan2(e2, e1)
axes[1].hist(np.degrees(phi), bins=36, range=(-90, 90),
             color='forestgreen', edgecolor='white')
axes[1].set_xlabel('Position angle (degrees)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Shape orientation (should be uniform)', fontsize=13)

plt.tight_layout()
plt.savefig('../../figures/ellipticity_properties.png', dpi=150, bbox_inches='tight')
plt.show()

print("A uniform position angle distribution means no preferred alignment,")
print("which is expected before lensing is applied.")
print("A non-uniform distribution would indicate PSF systematics.")

## Summary

In this notebook we:
1. Connected to the DP0.2 Butler
2. Retrieved the object catalog for a patch
3. Applied weak lensing quality cuts (isPrimary, galaxy, SNR, flags, blendedness)
4. Examined e1/e2 distributions (should be centered at ~0, RMS ~0.25)
5. Made a size-magnitude diagram showing star-galaxy separation
6. Checked for isotropy of shape orientations

**Next:** [02_psf_diagnostics.ipynb](02_psf_diagnostics.ipynb) — Validate the PSF model